# Example 11.3.1 / 2: Declare an Imported Function

Companion notebook for **11.3.1 Imported Functions and GSL** in the AMPL lab Documentation, section "Declare an Imported Function". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Declare an Imported Function

More detailed declarations can specify argument domains, symbolic or random return values, and
pipe functions. Arity, the number of arguments, may be fixed or variable according to the
imported implementation. A declared argument domain makes AMPL check each argument's membership,
but no declaration supplies derivatives or numerical safety that the library does not implement.
Use the declaration distributed with a library instead of guessing its signature. The library
must be loaded in each AMPL session before its functions are evaluated.

Each declaration form with AMPLGSL functions, evaluated with `display` or `call` after
`load amplgsl.dll;`:

| Form | Example | Result |
|---|---|---|
| plain numeric function | `function gsl_cdf_ugaussian_P;` | `gsl_cdf_ugaussian_P(1.96) = 0.975002` |
| declared but library not loaded | the same declaration without `load` | error `Can't invoke unavailable function gsl_cdf_ugaussian_P.` |
| fixed arity | `gsl_cdf_ugaussian_P(1, 2)` | error `gsl_cdf_ugaussian_P needs 1 argument(s), not 2` |
| argument domain | `function gsl_sf_gamma({1..10});` | `gsl_sf_gamma(5) = 24`; `gsl_sf_gamma(11)` fails with `11 is outside the domain of argument 1 to gsl_sf_gamma` |
| variable arity, direction word | `function gsl_sort(INOUT ...);` then `call gsl_sort({i in 1..4} d[i]);` | `d` changes from 3, 1, 4, 2 to 1, 2, 3, 4 |
| `random` | `function gsl_ran_gaussian random;` | two calls in one `display` return `0.3064` and `-0.424386`; declaring it without `random` warns `Function gsl_ran_gaussian is random.` |
| `symbolic` | `function gsl_version symbolic;` | `gsl_version() = 2.8`, a string result |
| `pipe` | `function mean2 pipe "awk '{print ($1+$2)/2}'";` | documented in the AMPL book as `mean2(1, 2) + 1 = 2.5`; it needs a Unix shell with `awk`, so it is not run here |

The variable-arity, `INOUT` and `call` rows as a checked example. The `call` statement has no
amplpy method, so the amplpy box sends it with `ampl.eval`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_sort(INOUT ...);  # sorts its argument list in place

param d {1..4};  # values to sort

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["d"] = {1: 3, 2: 1, 3: 4, 4: 2}

In [5]:
# run.py
ampl.eval("call gsl_sort({i in 1..4} d[i]);")
print(ampl.param["d"].to_dict())

{1: 1, 2: 2, 3: 3, 4: 4}


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param d := 1 3  2 1  3 4  4 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
call gsl_sort({i in 1..4} d[i]);
display d;

d [*] :=
1  1
2  2
3  3
4  4
;

